# Silver Layer - Retail Sales Lakehouse

**Purpose**: Cleanse, validate, and deduplicate data from Bronze layer

**Input Tables**:
- `bronze_sales`: Initial baseline data
- `bronze_incremental`: Incremental updates

**Output Tables**:
- `silver_sales`: Validated and deduplicated sales transactions
- `quarantine_schema`: Records with schema mismatches
- `quarantine_late`: Late-arriving transactions beyond threshold

**Job Parameters**:
- `late_threshold_days`: Maximum allowed delay for transaction data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# Read job parameter for late arrival threshold
dbutils.widgets.text("late_threshold_days", "")
late_threshold_days = int(dbutils.widgets.get("late_threshold_days"))

print(f"Late arrival threshold: {late_threshold_days} days")

In [0]:
# ====================================================
# 1. READ BRONZE TABLES
# ====================================================

# Read baseline Bronze table (initial historical load)
bronze_base_df = spark.table("bronze_sales")

# Read incremental Bronze table (ongoing updates)
bronze_incremental_df = spark.table("bronze_incremental")

# Cache counts to avoid multiple Spark actions
base_rows = bronze_base_df.count()
incoming_rows = bronze_incremental_df.count()

print(f"Bronze Base rows: {base_rows}")
print(f"Bronze Incremental rows: {incoming_rows}")

# ====================================================
# 2. CREATE SILVER TABLE IF MISSING
# ====================================================

# Check if silver_sales table exists
silver_exists = spark.catalog.tableExists("silver_sales")

if not silver_exists:
    print("Silver table does not exist. Creating from bronze_sales...")
    
    # Create silver_sales from baseline data
    bronze_base_df.write.format("delta").mode("overwrite").saveAsTable("silver_sales")
    
    print(f"Created silver_sales with {base_rows} initial rows")
    
    # Set flag to ensure schema evolution runs on first incremental load
    silver_exists = True
else:
    print("Silver table already exists")

In [0]:
# ====================================================
# 3. SCHEMA VALIDATION
# ====================================================

# Initialize new_columns to avoid NameError
new_columns = set()

# Define required business columns (metadata columns are ignored)
required_business_columns = [
    "transaction_id",
    "store_id",
    "customer_id",
    "product_category",
    "payment_method",
    "sales_amount",
    "transaction_time",
    "ingestion_time"
]

# Extract column names from both tables
base_columns = set(bronze_base_df.columns)
incremental_columns = set(bronze_incremental_df.columns)

# Check for missing required business columns in incremental batch
missing_business_columns = set(required_business_columns) - incremental_columns

if missing_business_columns:
    # Schema validation failed - quarantine entire batch
    print(f"❌ Schema validation FAILED. Missing business columns: {missing_business_columns}")
    
    # Write entire incremental batch to quarantine_schema using MERGE (idempotent)
    if not spark.catalog.tableExists("quarantine_schema"):
        bronze_incremental_df.write.format("delta").mode("overwrite").saveAsTable("quarantine_schema")
        print(f"Created quarantine_schema with {incoming_rows} rows")
    else:
        # MERGE into existing quarantine_schema table
        quarantine_schema_table = DeltaTable.forName(spark, "quarantine_schema")
        
        quarantine_schema_table.alias("quarantine").merge(
            bronze_incremental_df.alias("incoming"),
            "quarantine.transaction_id = incoming.transaction_id"
        ).whenMatchedUpdate(
            condition="incoming.ingestion_time > quarantine.ingestion_time",
            set={
                col: f"incoming.{col}" for col in bronze_incremental_df.columns
            }
        ).whenNotMatchedInsertAll().execute()
        
        print(f"Quarantined {incoming_rows} rows to quarantine_schema")
    
    # Stop execution - do not proceed to merge
    dbutils.notebook.exit("Schema validation failed. Batch quarantined.")
else:
    print("✅ Schema validation PASSED")
    
    # Identify new columns (schema evolution)
    new_columns = incremental_columns - base_columns
    if new_columns:
        print(f"New columns detected (schema evolution): {new_columns}")

In [0]:
# ====================================================
# 4. MANUAL SCHEMA EVOLUTION
# ====================================================

if silver_exists and new_columns:
    print("Detecting new columns for schema evolution...")
    
    # Get current Silver table columns
    silver_df = spark.table("silver_sales")
    silver_columns = set(silver_df.columns)
    
    # Find columns that need to be added
    columns_to_add = new_columns - silver_columns
    
    if columns_to_add:
        print("\nSchema Evolution Applied")
        print("Added Columns:")
        
        # Get schema once before loop to avoid repeated Analyze RPC calls
        incremental_schema = {field.name: field.dataType.simpleString() for field in bronze_incremental_df.schema.fields}
        
        # Add each new column with its datatype from incremental batch
        for col_name in columns_to_add:
            col_type = incremental_schema[col_name]
            
            # ALTER TABLE to add column (Community Edition compatible)
            spark.sql(f"ALTER TABLE silver_sales ADD COLUMNS ({col_name} {col_type})")
            print(f"  {col_name}")
    else:
        print("No new columns to add")

# -----------------------------------------------------
# ENTERPRISE IMPLEMENTATION (commented for interview)
# -----------------------------------------------------
# In Databricks Enterprise, you can use automatic schema evolution:
# 
# spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
# 
# This automatically handles new columns during MERGE operations
# without manual ALTER TABLE statements.
# -----------------------------------------------------

In [0]:
# ====================================================
# 5. DEDUPLICATE INCREMENTAL BATCH
# ====================================================

# Incoming batch may contain duplicate transaction_id values
# Keep only the newest record based on ingestion_time

print("Deduplicating incremental batch...")

# Add row number partitioned by transaction_id, ordered by ingestion_time DESC
window_spec = Window.partitionBy("transaction_id").orderBy(F.col("ingestion_time").desc())

incremental_with_rownum = bronze_incremental_df.withColumn(
    "row_num", 
    F.row_number().over(window_spec)
)

# Keep only the first row (newest) for each transaction_id
incremental_deduped = incremental_with_rownum.filter(F.col("row_num") == 1).drop("row_num")

deduped_rows = incremental_deduped.count()
duplicate_count = incoming_rows - deduped_rows
print(f"Removed {duplicate_count} duplicate rows")
print(f"Deduplicated batch size: {deduped_rows} rows")

In [0]:
# ====================================================
# 6. LATE ARRIVING DATA DETECTION
# ====================================================

# Calculate how late each transaction is (deterministic)
# Based on ingestion_time vs transaction_time, not current_date()
incremental_with_delay = incremental_deduped.withColumn(
    "days_late",
    F.datediff(
        F.to_date(F.col("ingestion_time")),
        F.to_date(F.col("transaction_time"))
    )
)

# Separate late vs valid records
late_records = incremental_with_delay.filter(F.col("days_late") > late_threshold_days)
valid_records = incremental_with_delay.filter(F.col("days_late") <= late_threshold_days).drop("days_late")

late_count = late_records.count()
valid_count = valid_records.count()

print(f"Late records (> {late_threshold_days} days): {late_count}")
print(f"Valid records: {valid_count}")

# Write late records to quarantine_late using MERGE (idempotent)
if late_count > 0:
    # Create quarantine_late table if it doesn't exist
    if not spark.catalog.tableExists("quarantine_late"):
        late_records.write.format("delta").mode("overwrite").saveAsTable("quarantine_late")
        print(f"Created quarantine_late with {late_count} rows")
    else:
        # MERGE into existing quarantine_late table
        quarantine_table = DeltaTable.forName(spark, "quarantine_late")
        
        quarantine_table.alias("quarantine").merge(
            late_records.alias("incoming"),
            "quarantine.transaction_id = incoming.transaction_id"
        ).whenMatchedUpdate(
            condition="incoming.ingestion_time > quarantine.ingestion_time",
            set={
                col: f"incoming.{col}" for col in late_records.columns
            }
        ).whenNotMatchedInsertAll().execute()
        
        print(f"Quarantined {late_count} late-arriving rows")

In [0]:
# ====================================================
# 7. MERGE VALID RECORDS INTO SILVER
# ====================================================

if valid_count > 0:
    print("Merging valid records into silver_sales...")
    
    # Load Silver table as DeltaTable for MERGE operation
    silver_table = DeltaTable.forName(spark, "silver_sales")
    
    # Perform idempotent MERGE
    # - Match on transaction_id
    # - Update only if incoming record is newer (based on ingestion_time)
    # - Insert if not matched
    
    silver_table.alias("silver").merge(
        valid_records.alias("incoming"),
        "silver.transaction_id = incoming.transaction_id"
    ).whenMatchedUpdate(
        condition="incoming.ingestion_time > silver.ingestion_time",
        set={
            col: f"incoming.{col}" for col in valid_records.columns
        }
    ).whenNotMatchedInsertAll().execute()
    
    print("✅ MERGE completed successfully")
else:
    print("No valid records to merge")

In [0]:
# ====================================================
# 8. LOGGING AND METRICS
# ====================================================

silver_df = spark.table("silver_sales")
silver_rows = silver_df.count()

print("\n" + "="*56)
print("Silver Pipeline Completed")
print("="*56)
print(f"Incoming Rows           : {incoming_rows:,}")
print(f"Duplicate Rows Removed  : {duplicate_count:,}")
print(f"Late Rows Quarantined   : {late_count:,}")
print(f"Valid Rows Merged       : {valid_count:,}")
print(f"Silver Row Count        : {silver_rows:,}")
print("="*56)

In [0]:
# ====================================================
# 9. DISPLAY RESULTS
# ====================================================

print("\n📊 Silver Sales Table:")
display(silver_df.orderBy(F.col("ingestion_time").desc()).limit(100))

if spark.catalog.tableExists("quarantine_late"):
    quarantine_df = spark.table("quarantine_late")
    if quarantine_df.count() > 0:
        print("\n⚠️ Quarantined Late Arrivals:")
        display(quarantine_df.orderBy(F.col("days_late").desc()))